# Sinh mô tả bằng mô hình thị giác–ngôn ngữ

Sinh mô tả chung, theo giải phẫu và theo khung lâm sàng bằng Qwen hoặc LLaVA.

In [ ]:
import os
import sys
from pathlib import Path

source_root = Path(os.environ.get("MEDVQA_SOURCE_ROOT", ".")).resolve()
if not (source_root / "medvqa").is_dir():
    raise FileNotFoundError("Set MEDVQA_SOURCE_ROOT to the extracted source directory.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

## Cấu hình thí nghiệm

Chọn mô hình, thư mục dữ liệu và nơi lưu mô tả. Dùng `PILOT` để kiểm tra kỹ thuật trước khi chạy `TEST` với cấu hình cố định.

In [ ]:
from pathlib import Path

MODE = "PILOT"
RUN_ID = "pilot_obs_Q"
OBSERVER = "Q"
ARMS = ["ARM-GEN", "ARM-ANA", "ARM-GRD"]
PLATFORM = None
DESIGN = "MAIN"
INPUT_ROOT = None
LOCK_FILE = None
RUN_PARAMS_FILE = None
PRIOR_OUTPUT_DIRS = []
OUT_ROOT = None
SHARD_TIME_LIMIT_S = None
SHARD_INDEX, NUM_SHARDS = (0, 1)
MAX_ITEMS_THIS_SESSION = None
PRECISION = "PREC-0"
OBS_MAX_NEW_TOKENS = 700
BACKEND = "hf"
LLAVAMED_CODE_COMMIT = None
LLAVAMED_CONV_MODE = None
INSTALL = False
MODULE_DIR = source_root

## Thực thi và lưu kết quả

In [ ]:
import sys
import time

sys.path.insert(0, str(MODULE_DIR))

In [ ]:
from medvqa import common as U
from medvqa import execution as RUN

PLATFORM = PLATFORM or RUN.MODEL_PLATFORM[OBSERVER]
PLATFORM_CFG = RUN.platform_setup(PLATFORM)
INPUT_ROOT = Path(INPUT_ROOT or PLATFORM_CFG["input_root"])
OUT_ROOT = Path(OUT_ROOT or PLATFORM_CFG["out_root"])
SHARD_TIME_LIMIT_S = SHARD_TIME_LIMIT_S or PLATFORM_CFG["time_limit_s"]
LOCK_FILE = Path(LOCK_FILE or INPUT_ROOT / "." / "locked_inputs.sha256")
RUN_PARAMS_FILE = Path(RUN_PARAMS_FILE or INPUT_ROOT / "." / "run_params.json")
if DESIGN == "MAIN" and OBSERVER == "M":
    raise ValueError("MAIN: OBS-M (LLaVA-Med) is not run")
ENV = RUN.env_for(OBSERVER, PLATFORM)
if OBSERVER == "M" and BACKEND == "hf" and (not LLAVAMED_CONV_MODE):
    raise ValueError(
        "Set LLAVAMED_CONV_MODE from the official eval script at the pinned commit"
    )
RUN.verify_run_params_first(LOCK_FILE, RUN_PARAMS_FILE, MODE)
platform_report = RUN.check_platform(OBSERVER, PLATFORM, MODE, RUN_PARAMS_FILE)
RUN.check_design(DESIGN, MODE, RUN_PARAMS_FILE)
ENV_PINS = RUN.env_pins_for(MODE, RUN_PARAMS_FILE, ENV)
if INSTALL and BACKEND == "hf":
    RUN.install_env(ENV, ENV_PINS, LLAVAMED_CODE_COMMIT)
env_report = RUN.verify_env(
    ENV, ENV_PINS, MODE, LLAVAMED_CODE_COMMIT, check_git=BACKEND == "hf"
)
print("environment:", env_report)

In [ ]:
from medvqa import conditions as CL
from medvqa import models as B
from medvqa import prompts as P
from medvqa import routing as R

PREP = INPUT_ROOT / "prep_out"
PROMPTS = INPUT_ROOT / "resources/protocol" / "prompts"
U.assert_no_sealed_inputs([INPUT_ROOT] + list(PRIOR_OUTPUT_DIRS))

In [ ]:
COHORTS = RUN.cohorts_for(MODE)
lock_files = RUN.prompt_lock_files(INPUT_ROOT)
lock_files.update(
    RUN.prep_lock_files(
        INPUT_ROOT,
        [
            "route_metadata.json",
            "route_pairs.json",
            "observer_prompt_manifest.json",
            "images_index.json",
        ]
        + [f"cohorts/{c}.jsonl" for c in COHORTS],
    )
)
if MODE == "TEST":
    lock_files["run_params.json"] = RUN_PARAMS_FILE
lock_report = U.verify_locked_inputs(LOCK_FILE, lock_files, MODE)
if MODE == "TEST":
    rp = U.read_json(RUN_PARAMS_FILE)
    if (
        RUN.rp_get(rp, "precision", OBSERVER) != PRECISION
        or RUN.rp_get(rp, "obs_max_new_tokens") != OBS_MAX_NEW_TOKENS
    ):
        raise U.LockError(
            "PRECISION / OBS_MAX_NEW_TOKENS differ from locked run_params.json"
        )
    if OBSERVER == "M" and (
        RUN.rp_get(rp, "llavamed", "code_commit") != LLAVAMED_CODE_COMMIT
        or RUN.rp_get(rp, "llavamed", "conv_mode") != LLAVAMED_CONV_MODE
    ):
        raise U.LockError(
            "LLaVA-Med commit/conv_mode differ from locked run_params.json"
        )
print(
    "locked inputs verified:",
    len(lock_report["files"]),
    "| not yet locked:",
    len(lock_report["unlisted_used"]),
)

In [ ]:
rows = RUN.load_cohort_rows(PREP, COHORTS)
nids = sorted({r["neutral_id"] for r in rows})
cohort_of = {r["neutral_id"]: r["cohort"] for r in rows}
rm = U.read_json(PREP / "route_metadata.json")["records"]
rp_table = U.read_json(PREP / "route_pairs.json")
routing = {n: R.route_for(n, rm[n], rp_table) for n in nids}
manifest = U.read_json(PREP / "observer_prompt_manifest.json")["entries"]
index = U.read_json(PREP / "images_index.json")
bank = P.ObserverPromptBank(PROMPTS)
allowed = bank.allowed_texts(sorted({r["route_key"] for r in routing.values()}))
locked_lines = bank.locked_lines()
questions = sorted({r["question"] for r in rows})
options = sorted({o["text"] for r in rows if r["options"] for o in r["options"]})
OBS_ID = CL.OBSERVERS[OBSERVER]
jobs = [(OBS_ID, arm, n) for n in nids for arm in ARMS]
keys = [f"OBS|{o}|{a}|{n}" for o, a, n in jobs]
shard_keys = U.plan_shard(keys, NUM_SHARDS, SHARD_INDEX)
print(
    MODE,
    COHORTS,
    "| images:",
    len(nids),
    "| jobs:",
    len(keys),
    "| this shard:",
    len(shard_keys),
)

In [ ]:
if BACKEND == "stub":
    model = B.StubVLM(OBSERVER, PRECISION)
elif OBSERVER == "M":
    model = B.LlavaMedBackend(PRECISION, LLAVAMED_CONV_MODE)
else:
    model = B.HFVisionBackend(OBSERVER, PRECISION)
session = U.ShardSession(
    "obs_open",
    RUN_ID,
    MODE,
    OUT_ROOT,
    SHARD_INDEX,
    NUM_SHARDS,
    PRIOR_OUTPUT_DIRS,
    time_limit_s=SHARD_TIME_LIMIT_S,
    max_items=MAX_ITEMS_THIS_SESSION,
)
run_cfg = {
    "notebook": "obs_open",
    "platform": dict(PLATFORM_CFG, lock=platform_report),
    "design": DESIGN,
    "mode": MODE,
    "observer": OBS_ID,
    "arms": ARMS,
    "cohorts": COHORTS,
    "model": model.identity,
    "precision": PRECISION,
    "max_new_tokens": OBS_MAX_NEW_TOKENS,
    "generate_kwargs": B.GREEDY_KW,
    "generation_config_effective": model.identity.get("generation_config_effective"),
    "system_prompt": "none (chat template default)",
    "image_position": "before text",
    "qwen_pixels": [B.QWEN_MIN_PIXELS, B.QWEN_MAX_PIXELS] if OBSERVER == "Q" else None,
    "llavamed": {"code_commit": LLAVAMED_CODE_COMMIT, "conv_mode": LLAVAMED_CONV_MODE}
    if OBSERVER == "M"
    else None,
    "env": U.environment_info(),
    "env_pins": env_report,
    "locked_inputs": lock_report,
    "shard": [SHARD_INDEX, NUM_SHARDS],
}
run_cfg_sha = RUN.write_run_config(session, run_cfg)
print("session:", session.out_dir)

In [ ]:
done, _ = session.done_keys()
todo = [(k, j) for k, j in zip(keys, jobs) if k in set(shard_keys) and k not in done]
print("already done:", len(shard_keys) - len(todo), "| to do:", len(todo))
for key, (obs_id, arm, nid) in todo:
    if session.should_stop():
        break
    route = routing[nid]
    prompt, source = bank.prompt(arm, route["routes"])
    P.check1_observer_prompt(
        prompt, allowed, questions, options, locked_lines=locked_lines
    )
    P.check3_no_original_names(prompt)
    if manifest[f"{arm}|{nid}"]["prompt_sha256"] != U.sha256_text(prompt):
        raise U.IntegrityError(f"prompt differs from prep manifest for {arm}|{nid}")
    rec = session.base("OBS-v1", key)
    rec.update(
        {
            "observer": obs_id,
            "arm": arm,
            "neutral_id": nid,
            "cohort": cohort_of[nid],
            "route_key": route["route_key"],
            "routes": route["routes"],
            "route_gaps": route["gaps"],
            "prompt_source": source,
            "prompt_sha256": U.sha256_text(prompt),
            "attempts": [{"attempt": 1}],
            "gen": {
                "max_new_tokens": OBS_MAX_NEW_TOKENS,
                "do_sample": False,
                "precision": PRECISION,
            },
        }
    )
    image = RUN.open_image(PREP, nid)
    if image is None:
        rec.update(
            {
                "status": "image_broken",
                "text": "",
                "n_output_tokens": 0,
                "truncated": False,
                "finish_reason": "none",
            }
        )
        session.write(rec)
        continue
    t0 = time.time()
    try:
        g = model.generate(image, prompt, OBS_MAX_NEW_TOKENS)
    except Exception as exc:
        rec.update(
            {
                "status": "error",
                "error": f"{type(exc).__name__}: {str(exc)[:300]}",
                "text": "",
            }
        )
        session.write(rec)
        continue
    rec.update(
        {
            "status": "ok",
            "text": g["text"],
            "n_output_tokens": g["n_output_tokens"],
            "truncated": g["truncated"],
            "finish_reason": g["finish_reason"],
            "first_step_finite": g["first_step_finite"],
            "quality": U.quality_flags(g["text"]),
            "timing_s": round(time.time() - t0, 3),
        }
    )
    session.write(rec)

In [ ]:
mine = [r for r in U.read_jsonl(session.records) if r["status"] == "ok"]
allrows = list(
    RUN.load_records(
        [session.out_dir.parent] + list(PRIOR_OUTPUT_DIRS), "OBS-v1"
    ).values()
)
quality = U.summarize_quality(allrows, ("observer", "arm", "route_key"))
n = max(1, len(allrows))
tech = {
    "n": len(allrows),
    "nonfinite_first_step": sum((r.get("first_step_finite") is False for r in allrows)),
    "empty_rate": sum((r["quality"]["empty"] for r in allrows if "quality" in r)) / n,
    "repeat_char20_rate": sum(
        (r["quality"]["repeat_char20"] for r in allrows if "quality" in r)
    )
    / n,
    "max_truncated_rate_cell": max(
        [q["truncated_rate"] for q in quality.values()] or [0]
    ),
    "median_timing_s": sorted((r.get("timing_s", 0) for r in allrows))[
        len(allrows) // 2
    ]
    if allrows
    else None,
}
tech["escalate_precision"] = (
    tech["nonfinite_first_step"] > 0
    or tech["empty_rate"] > 0.01
    or tech["repeat_char20_rate"] > 0.01
)
tech["raise_cap_to_900"] = tech["max_truncated_rate_cell"] > 0.02
rep = session.integrity(
    shard_keys,
    {
        "checks": {
            "check1": "pass",
            "check3": "pass",
            "check4": "pass",
            "check7": lock_report["verified"],
            "prompt_manifest_sha": "pass",
        },
        "quality": quality,
        "technical": tech,
        "run_config_sha256": run_cfg_sha,
        "locked_inputs": {
            "verified": lock_report["verified"],
            "unlisted_used": lock_report["unlisted_used"],
        },
    },
)
if MODE == "PILOT":
    U.write_json(session.out_dir / f"precision_report_{OBS_ID}.json", tech)
print(
    {
        k: rep[k]
        for k in (
            "stopped_reason",
            "done_this_session",
            "done_total_in_shard",
            "remaining_in_shard",
            "errors",
            "duplicate_keys",
        )
    }
)
print("technical:", tech)